In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG   = "clutchlytics"
SCHEMA    = "bronze"
VOLUME    = "nhl_raw"
TABLE     = f"{CATALOG}.{SCHEMA}.raw_nhl_games"
 
# ← Update FILE_NAME for each new round upload
FILE_NAME = "round_1_game_ids.json"
 
FILE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FILE_NAME}"
 
print(f"Source : {FILE_PATH}")
print(f"Target : {TABLE}")

In [0]:
# ── CONFIRM FILE EXISTS ───────────────────────────────────────────────────────
 
files      = dbutils.fs.ls(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
file_names = [f.name for f in files]
 
if FILE_NAME not in file_names:
    raise FileNotFoundError(
        f"'{FILE_NAME}' not found in Volume.\n"
        f"Files available: {file_names}"
    )
 
file_size = [f.size for f in files if f.name == FILE_NAME][0]
print(f"File confirmed: {FILE_NAME} ({file_size:,} bytes)")

In [0]:
# ── READ + PARSE JSON ────────────────────────────────────────────────────────
 
import json
from datetime import datetime, timezone
 
raw_text   = spark.read.text(FILE_PATH)
json_str   = "\n".join([row.value for row in raw_text.collect()])
payload    = json.loads(json_str)
 
# ── Extract meta envelope ──
meta        = payload.get("meta", {})
pulled_at   = meta.get("pulled_at")
season      = meta.get("season")
season_type = meta.get("season_type")
round_num   = meta.get("round")
source_url  = meta.get("source_url")
endpoint    = meta.get("endpoint")
 
# ── Extract data block ──
data       = payload.get("data", {})
games      = data.get("games", [])
game_count = data.get("game_count")
raw_count  = data.get("raw_game_count")
excluded   = data.get("excluded_games", [])
 
print(f"Meta:")
print(f"  season      : {season}")
print(f"  season_type : {season_type}")
print(f"  round       : {round_num}")
print(f"  endpoint    : {endpoint}")
print(f"  pulled_at   : {pulled_at}")
print(f"\nData:")
print(f"  raw_game_count      : {raw_count}")
print(f"  game_count (clean)  : {game_count}")
print(f"  excluded_games      : {len(excluded)}")
print(f"  games in array      : {len(games)}")

In [0]:
# ── BUILD ROWS ────────────────────────────────────────────────────────────────
# One row per game. All fields from game object + meta context columns.
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
rows = []
for game in games:
    rows.append({
        # ── Game identity ──
        "event_id":     game.get("event_id"),
        "game_name":    game.get("name"),
        "short_name":   game.get("short_name"),
        "game_date":    game.get("date"),
        "status":       game.get("status"),
        "completed":    game.get("completed"),
        "venue":        game.get("venue"),
 
        # ── Home team ──
        "home_team_id": game.get("home_team_id"),
        "home_team":    game.get("home_team"),
        "home_score":   game.get("home_score"),
        "home_winner":  game.get("home_winner"),
 
        # ── Away team ──
        "away_team_id": game.get("away_team_id"),
        "away_team":    game.get("away_team"),
        "away_score":   game.get("away_score"),
        "away_winner":  game.get("away_winner"),
 
        # ── Season / playoff context (from meta) ──
        "season":       season,
        "season_type":  season_type,
        "round":        round_num,
 
        # ── Ingestion metadata ──
        "source_file":  FILE_NAME,
        "source_url":   source_url,
        "pulled_at":    pulled_at,
        "ingested_at":  ingested_at,
    })
 
print(f"Rows built: {len(rows)}")
print(f"\nSample — first game:")
for k, v in rows[0].items():
    print(f"  {k:<20} = {v}")

In [0]:
# ── WRITE TO DELTA ────────────────────────────────────────────────────────────
# MERGE on event_id — prevents duplicates if same file is re-uploaded.
# Append-friendly — future rounds land as new rows in the same table.
 
games_df = spark.createDataFrame(rows)
 
table_exists = spark.catalog.tableExists(TABLE)
 
if not table_exists:
    (
        games_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE)
    )
    print(f"Table created: {TABLE}")
 
else:
    games_df.createOrReplaceTempView("new_games")
 
    spark.sql(f"""
        MERGE INTO {TABLE} AS target
        USING new_games AS source
        ON target.event_id = source.event_id
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    print(f"Merged into existing table: {TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
result = spark.sql(f"""
    SELECT
        event_id,
        short_name,
        game_date,
        home_team,
        home_score,
        away_team,
        away_score,
        completed,
        season_type,
        round
    FROM {TABLE}
    ORDER BY game_date
""")
 
total = result.count()
print(f"Total rows in {TABLE}: {total}")
result.show(50, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                        AS total_games,
        COUNT(DISTINCT round)                           AS rounds,
        COUNT(CASE WHEN completed = true  THEN 1 END)  AS completed_games,
        COUNT(CASE WHEN completed = false THEN 1 END)  AS incomplete_games,
        COUNT(DISTINCT home_team_id)                   AS unique_home_teams,
        COUNT(CASE WHEN event_id IS NULL  THEN 1 END)  AS null_event_ids,
        COUNT(CASE WHEN home_score IS NULL
                    AND completed = true THEN 1 END)   AS missing_scores,
        MIN(game_date)                                 AS earliest_game,
        MAX(game_date)                                 AS latest_game
    FROM {TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)